# HORM Colab GPU runner (cymek-beta)

**Runtime: GPU (any Colab GPU runtime).** Runtime -> Change runtime type
-> **GPU** -> Run all. Expects ~30-60 min, ends with an auto-download.

What runs, in order (all commands are committed repo code, unmodified):
0. Cell 0 fetches and hard-resets to the branch tip, so this notebook
   always runs the latest code (download any prior results first).
1. Fast gate: hormonal unit/integration/session tests (seconds).
2. HORM-003 prospective A/B (`--horm003 --force`: prior committed result
   is archived to `.previous`, never silently overwritten).
3. HORM-004 live-appraisal A/B (`--horm004 --force`, same archive rule).
4. Suite in OOM-proof chunks (bulk, R1C driver, production-entry
   per-test) with per-stage logs and auto-printed failure tails.
   The 11 large-campaign tests (each peaks near 10 GiB, measured)
   run only behind a 14 GiB RAM gate; smaller hosts record them as
   explicit RAM exclusions instead of dying.
5. Closure receipt regenerated from measured junit counts, then gated.
6. Packaging: every RESULT/manifest hashed, bundle zipped for download.

Head commit is recorded into the bundle at runtime for provenance.


In [ ]:
# CELL 0: FREEZE repo + environment (fail closed)
import hashlib
import os
import subprocess
import sys

REPO = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
BRANCH = "cymek-beta"
REPO_DIR = "/content/repo"

import os.path
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, REPO_DIR],
                   check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", "origin/" + BRANCH],
               check=True)
subprocess.run(["git", "status", "--short"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers", "pytest"], check=True)

import torch
assert torch.cuda.is_available(), "HORM Colab run requires Google Colab GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

HEAD_SHA = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                          capture_output=True, text=True).stdout.strip()
print("HEAD_SHA:", HEAD_SHA)

def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SPEC_SHA = _sha256_file("v5_contracts/model_spec.py")
GATE_SHA = _sha256_file("artifacts/v5/launch_readiness.json")
assert SPEC_SHA.upper().startswith("DFDCD883"), \
    "frozen model_spec.py drifted: " + SPEC_SHA
assert GATE_SHA.upper().startswith("95B2331A"), \
    "launch_readiness.json drifted: " + GATE_SHA
print("frozen model_spec.py + launch_readiness.json: VERIFIED")
print("HORM COLAB PREEXECUTION GATE: PASS")


In [ ]:
# CELL 1: RUN (committed commands only; --force archives prior results)
import os
import subprocess
import sys

# Preregistered CPU-only execution: hide CUDA so the HORM runners (which
# refuse to run with visible CUDA) execute exactly as designed. Heavy
# stages run in separate fresh processes (one 700-test process is
# OOM-killed on Colab-free RAM); every stage streams to the cell output
# AND to /content/horm-logs for post-mortem. Thread caps per stage: the
# wall-clock-sensitive R1C driver gets 4 threads, everything else 2.
BASE_ENV = dict(os.environ, PYTHONPATH="/content/repo",
                CUDA_VISIBLE_DEVICES="", ANRA_TEST_DEVICE="cpu")
LOGDIR = "/content/horm-logs"
os.makedirs(LOGDIR, exist_ok=True)
STAGE = [0]

def run(*args, threads="2", check=True):
    STAGE[0] += 1
    log = "%s/stage%02d.log" % (LOGDIR, STAGE[0])
    env = dict(BASE_ENV, OMP_NUM_THREADS=threads,
               MKL_NUM_THREADS=threads)
    print("\n=== ", " ".join(args),
          " [threads=%s log=%s] ===" % (threads, log), flush=True)
    handle = open(log, "w", encoding="utf-8", errors="replace")
    proc = subprocess.Popen(
        [sys.executable, "-u", *args], env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors="replace")
    for line in proc.stdout:
        sys.stdout.write(line)
        handle.write(line)
    proc.wait()
    handle.close()
    if check and proc.returncode != 0:
        tail = open(log, encoding="utf-8", errors="replace").read()
        tail = tail.splitlines()
        print("---- tail of %s ----" % log)
        for tline in tail[-40:]:
            print(tline)
        raise SystemExit("STAGE FAILED (exit %s), full log: %s"
                         % (proc.returncode, log))
    return proc.returncode

def run_each(label, node_ids, threads="2"):
    bad = []
    for index, node in enumerate(node_ids):
        xml = "%s/%s_%02d.xml" % (LOGDIR, label, index)
        rc = run(*PYTEST, node, "--junitxml", xml,
                 threads=threads, check=False)
        if rc != 0 or not os.path.isfile(xml):
            bad.append(node)
    if bad:
        print("KILLED OR FAILED (%d):" % len(bad))
        for node in bad:
            print("  " + node)
        raise SystemExit("PE stage %s has %d bad tests" % (label,
                                                         len(bad)))

PYTEST = ["-m", "pytest", "-q", "-rf", "-p", "no:cacheprovider"]
run(*PYTEST, "tests/test_hormonal_state.py",
    "tests/test_hormonal_integration.py",
    "tests/test_hormonal_session.py", threads="2")
run("experiments/HORM-001/run_horm002_ab.py", "--horm003",
    "--output", "experiments/HORM-001", "--force", threads="2")
run("experiments/HORM-001/run_horm002_ab.py", "--horm004",
    "--output", "experiments/HORM-001", "--force", threads="2")
run(*PYTEST, "tests",
    "--ignore=tests/test_production_entry.py",
    "--ignore=tests/test_v5_cyr_gpu014_r1c_e2e_preflight.py",
    "--junitxml", "%s/bulk.xml" % LOGDIR, threads="2")
run(*PYTEST, "tests/test_v5_cyr_gpu014_r1c_e2e_preflight.py",
    "--junitxml", "%s/r1c.xml" % LOGDIR, threads="4")
PE_NODES = [
"tests/test_production_entry.py::test_exact_completion_single_partial_update",
"tests/test_production_entry.py::test_fresh_determinism_same_inputs_same_receipt",
"tests/test_production_entry.py::test_pack_smaller_than_budget_fails_closed",
"tests/test_production_entry.py::test_mid_campaign_resume_matches_uninterrupted",
"tests/test_production_entry.py::test_resume_changed_documents_fails",
"tests/test_production_entry.py::test_resume_changed_seed_fails",
"tests/test_production_entry.py::test_resume_changed_budget_fails",
"tests/test_production_entry.py::test_cross_store_and_cross_lineage_rejected",
"tests/test_production_entry.py::test_full_update_uses_four_microsteps_one_step",
"tests/test_production_entry.py::test_bucket_shapes_exact_and_certified",
"tests/test_production_entry.py::test_partial_tail_never_overshoots",
"tests/test_production_entry.py::test_single_microstep_update_accounts_exactly",
"tests/test_production_entry.py::test_milestone_checkpoints_published_and_receipted",
"tests/test_production_entry.py::test_milestone_protection_across_sessions",
"tests/test_production_entry.py::test_dangling_milestone_detected",
"tests/test_production_entry.py::test_rotation_keeps_milestones_and_head",
"tests/test_production_entry.py::test_recovery_cadence_publishes",
"tests/test_production_entry.py::test_lr_matches_canonical_schedule",
"tests/test_production_entry.py::test_lr_no_rewarm_after_resume",
"tests/test_production_entry.py::test_precision_receipt_matches_device",
"tests/test_production_entry.py::test_tpu_runtime_fails_closed",
"tests/test_production_entry.py::test_xla_execution_fails_closed_without_hardware",
"tests/test_production_entry.py::test_production_requires_contamination_commitment",
"tests/test_production_entry.py::test_production_requires_mixture_and_freeze",
"tests/test_production_entry.py::test_raw_source_required_in_production",
"tests/test_production_entry.py::test_contamination_content_binding",
"tests/test_production_entry.py::test_development_mode_labels_and_relaxes",
"tests/test_production_entry.py::test_frozen_mixture_end_to_end",
"tests/test_production_entry.py::test_mixture_shortfall_fails_closed",
"tests/test_production_entry.py::test_cognition_mixture_resume_equality",
"tests/test_production_entry.py::test_epoch_replay_when_permitted",
"tests/test_production_entry.py::test_campaign_certificate_completeness",
"tests/test_production_entry.py::test_banned_symbols_absent_from_entry_source",
"tests/test_production_entry.py::test_frozen_topology_multiplies",
"tests/test_production_entry.py::test_frozen_mixture_matches_contract",
"tests/test_production_entry.py::test_microstep_buckets_follow_supercycle",
"tests/test_production_entry.py::test_crossed_milestones_pure",
"tests/test_production_entry.py::test_resolve_cymek_sha_rejects_and_resolves",
"tests/test_production_entry.py::test_prepare_data_deterministic",
"tests/test_production_entry.py::test_build_milestone_receipt_self_describing",
"tests/test_production_entry.py::test_eos_packing_contract",
"tests/test_production_entry.py::test_compressed_e2e_fresh_recovery_milestone_stop_resume_partial_complete",
"tests/test_production_entry.py::test_multi_session_soak_state_machine",
"tests/test_production_entry.py::test_session_completes_tiny_campaign",
"tests/test_production_entry.py::test_session_timebox_resumable_then_completes",
"tests/test_production_entry.py::test_v5a_exact_parameter_count",
"tests/test_production_entry.py::test_bf16_trajectory_diagnostic",
"tests/test_production_entry.py::test_durable_mirror_session_recovery",
"tests/test_production_entry.py::test_already_complete_short_circuits",
"tests/test_production_entry.py::test_freeze_production_identity",
"tests/test_production_entry.py::test_materialize_first_party_supply_accounting",
]
assert len(PE_NODES) == 51, len(PE_NODES)
PE_HEAVY_SUFFIXES = [
    "test_mid_campaign_resume_matches_uninterrupted",
    "test_full_update_uses_four_microsteps_one_step",
    "test_bucket_shapes_exact_and_certified",
    "test_milestone_protection_across_sessions",
    "test_rotation_keeps_milestones_and_head",
    "test_lr_no_rewarm_after_resume",
    "test_frozen_mixture_end_to_end",
    "test_cognition_mixture_resume_equality",
    "test_compressed_e2e_fresh_recovery_milestone_stop_resume_partial_complete",
    "test_multi_session_soak_state_machine",
    "test_session_timebox_resumable_then_completes",
]
PE_HEAVY = [node for node in PE_NODES
            if node.rsplit("::", 1)[-1] in PE_HEAVY_SUFFIXES]
PE_LIGHT = [node for node in PE_NODES if node not in PE_HEAVY]
assert len(PE_HEAVY) == 11, len(PE_HEAVY)
assert len(PE_LIGHT) == 40, len(PE_LIGHT)
RAM_GB = 0.0
try:
    with open("/proc/meminfo", encoding="utf-8") as _meminfo:
        for _line in _meminfo:
            if _line.startswith("MemAvailable:"):
                RAM_GB = int(_line.split()[1]) / (1024 * 1024)
                break
except OSError:
    pass
print("available RAM: %.1f GiB" % RAM_GB)
HEAVY_OK = RAM_GB >= 14.0
run_each("pe_light", PE_LIGHT, threads="2")
if HEAVY_OK:
    run_each("pe_heavy", PE_HEAVY, threads="2")
else:
    print("RAM GATE: 11 large-campaign tests peak near 10 GiB each"
          " (measured); this host is below the 14 GiB gate, so they"
          " are recorded as RAM-excluded, not run")
# Regenerate the closure receipt from MEASURED junit counts (sanctioned:
# re-run and regenerate, never hand-edit), then gate on the receipt test.
import glob
import xml.etree.ElementTree as ET
from v5_training.test_receipt import build_receipt, write_receipt

def _stage_stats(prefixes):
    passed = failed = skipped = 0
    nfiles = 0
    for prefix in prefixes:
        for path in sorted(glob.glob("%s/%s*.xml" % (LOGDIR, prefix))):
            suite = ET.parse(path).getroot().find("testsuite")
            get = lambda key: int(suite.get(key) or 0)
            failed += get("failures") + get("errors")
            skipped += get("skipped")
            passed += get("tests") - get("failures") - get("errors") \
                - get("skipped")
            nfiles += 1
    return {"passed": passed, "failed": failed,
            "skipped": skipped}, nfiles

HEAD_SHA = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                          capture_output=True, text=True).stdout.strip()
STAGES = [("bulk", "tests excluding production-entry and r1c-driver",
           ["bulk"]),
          ("r1c", "R1C driver preflight", ["r1c"]),
          ("pe_light", "production-entry light (40 tests)",
           ["pe_light"]),
          ("pe_heavy", "production-entry large-campaign (11 tests)",
           ["pe_heavy"])]
entries = []
for _name, _desc, _files in STAGES:
    if _name == "pe_heavy" and not HEAVY_OK:
        entries.append({"command": "colab stage pe_heavy: RAM-excluded",
                        "device": "cpu", "files": ["tests/"],
                        "provenance": "RAM_EXCLUDED: host RAM %.1f GiB"
                        " below the 14 GiB gate; single large-campaign"
                        " tests peak near 10 GiB (measured locally); 11"
                        " tests excluded, not run, not counted as pass"
                        % RAM_GB,
                        "passed": 0, "failed": 0, "skipped": 11})
        continue
    _counts, _nfiles = _stage_stats(_files)
    entries.append({"command": "colab stage %s: %s" % (_name, _desc),
                    "device": "cpu", "files": ["tests/"],
                    "provenance": "Colab T4 run, %d junit files"
                    " for stage %s" % (_nfiles, _name),
                    **_counts})
write_receipt("artifacts/v5/cymek_500m_closure_test_receipt.json",
              build_receipt(tested_commit_sha=HEAD_SHA, results=entries,
                            environment={"note": "Colab GPU-host CPU-only run; no scientific GPU or TPU training was performed by this receipt."}))
print("receipt regenerated at", HEAD_SHA)
run(*PYTEST, "tests/test_production_entry.py::test_exact_head_test_receipt",
    threads="2")
run("-m", "v5_contracts.import_boundaries", threads="2")
print("\nALL HORM COLAB STAGES COMPLETE")


In [ ]:
# CELL 2: PACKAGE + DOWNLOAD (hash-bound bundle)
import glob
import hashlib
import json
import os
import shutil
import subprocess

HEAD_SHA = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                          capture_output=True, text=True).stdout.strip()
manifest = {"head_sha": HEAD_SHA, "files": { }}
for path in sorted(glob.glob("experiments/HORM-001/RESULT*.json")):
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    document = json.load(open(path, encoding="utf-8"))
    manifest["files"][os.path.basename(path)] = {
        "sha256": digest,
        "verdict": document.get("verdict"),
        "result_sha256": document.get("sha256"),
    }
    print(os.path.basename(path), "->", document.get("verdict"))
with open("experiments/HORM-001/COLAB_BUNDLE_MANIFEST.json", "w",
           encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, sort_keys=True)
    handle.write("\n")
shutil.copytree("/content/horm-logs", "experiments/HORM-001/colab-logs",
                dirs_exist_ok=True)
shutil.make_archive("/content/HORM-COLAB-RESULTS", "zip",
                    "experiments/HORM-001")
print("bundle:", "/content/HORM-COLAB-RESULTS.zip")
try:
    from google.colab import files
    files.download("/content/HORM-COLAB-RESULTS.zip")
except Exception as exc:
    print("manual download from experiments/HORM-001/:", exc)
